## AlexNet with Galaxy10 DECals Dataset

The Galaxy10 DECals dataset consists of 17,736 256 $\times$ 256 images of galaxies classified into 10 categories. The underlying images were originally sourced from the _Sloan Digital Sky Survey (SDSS)_ via the Galaxy Zoo project. Here we use the dataset to train a galaxy type classifier based on AlexNet

In [ ]:
import torch
import torchvision
import torchvision.transforms.functional as TF
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import transforms
import numpy as np
from tqdm.notebook import trange
import h5py
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
print("Device:", device)

### Load the Data

In [ ]:
with h5py.File('Galaxy10_DECals.h5', 'r') as gF:
    images = np.array(gF['images'])
    labels = np.array(gF['ans'])

In [ ]:
from sklearn.utils import shuffle
images, labels = shuffle(images, labels)

### Prepare the Data

The data must be prepared in a similar fashion to the MNIST examples except now there are three channels instead of one. The classes must be converted to a categorical variable.

In [ ]:
train_idx, test_idx = train_test_split(np.arange(labels.shape[0]), test_size=0.1)
X_train, y_train, X_test, y_test = images[train_idx], labels[train_idx], images[test_idx], labels[test_idx]

encoder = OneHotEncoder(sparse_output=False)
train_y = encoder.fit_transform(y_train.reshape(-1, 1))
test_y = encoder.transform(y_test.reshape(-1, 1))

X_train = X_train.astype('float32')
X_test = X_test.astype('float32')
X_train /= 255
X_test /= 255

X_train = torch.Tensor(X_train).permute(0, 3, 1, 2)
train_y = torch.Tensor(train_y)
train_dataset = TensorDataset(X_train, train_y)

X_test = torch.Tensor(X_test).permute(0, 3, 1, 2)
test_y = torch.Tensor(test_y)
test_dataset = TensorDataset(X_test, test_y)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, 
                                          shuffle=False, drop_last=True)

In [ ]:
class CustomAugmentations:
    def __init__(self, rotation_range, width_shift_range, height_shift_range, shear_range, zoom_range, horizontal_flip):
        self.rotation_range = rotation_range
        self.width_shift_range = width_shift_range
        self.height_shift_range = height_shift_range
        self.shear_range = shear_range
        self.zoom_range = zoom_range
        self.horizontal_flip = horizontal_flip

    def __call__(self, img):
        # Random rotation
        angle = torch.empty(1).uniform_(-self.rotation_range, self.rotation_range).item()
        img = TF.rotate(img, angle, interpolation=TF.InterpolationMode.NEAREST)

        # Random width shift
        width_shift = torch.empty(1).uniform_(-self.width_shift_range, self.width_shift_range).item()
        img = TF.affine(img, angle=0, translate=(width_shift * img.shape[2], 0), scale=1, shear=0, interpolation=TF.InterpolationMode.NEAREST)

        # Random height shift
        height_shift = torch.empty(1).uniform_(-self.height_shift_range, self.height_shift_range).item()
        img = TF.affine(img, angle=0, translate=(0, height_shift * img.shape[1]), scale=1, shear=0, interpolation=TF.InterpolationMode.NEAREST)

        # Random shear
        shear = torch.empty(1).uniform_(-self.shear_range, self.shear_range).item()
        img = TF.affine(img, angle=0, translate=(0, 0), scale=1, shear=shear, interpolation=TF.InterpolationMode.NEAREST)

        # Random zoom
        zoom = torch.empty(1).uniform_(1 - self.zoom_range, 1 + self.zoom_range).item()
        img = TF.affine(img, angle=0, translate=(0, 0), scale=zoom, shear=0, interpolation=TF.InterpolationMode.NEAREST)

        # Random horizontal flip
        if self.horizontal_flip and torch.rand(1).item() > 0.5:
            img = TF.hflip(img)
        
        return img

custom_transforms = transforms.Compose([
    CustomAugmentations(rotation_range=40, 
                        width_shift_range=0.2, 
                        height_shift_range=0.2, 
                        shear_range=0.2, 
                        zoom_range=0.2, 
                        horizontal_flip=True)
])

class AugmentedDataset(torch.utils.data.Dataset):
    def __init__(self, dataset, transform=None):
        self.dataset = dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        x, y = self.dataset[idx]
        if self.transform:
            x = self.transform(x)
        return x, y

aug_train_dataset = AugmentedDataset(train_dataset, 
                                     transform=custom_transforms)

train_loader = DataLoader(aug_train_dataset, batch_size=128, shuffle=True)

In [ ]:
def train_model(model, train_loader, test_loader, loss_fn, optimizer, epochs, device):
    train_errors = []
    test_errors = []
    train_accuracies = []
    test_accuracies = []

    tqdm_epoch = trange(epochs)
    for epoch in tqdm_epoch:
        model.train()
        train_loss = 0.0
        correct_train = 0

        # Training
        for batch_X, batch_y in train_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)
            # Forward pass
            outputs = model(batch_X)
            loss = loss_fn(outputs.squeeze(), batch_y)
            
            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * batch_X.size(0)
            # Calculate accuracy
            predicted = torch.argmax(outputs.squeeze(), dim=1)
            targets = torch.argmax(batch_y, dim=1)
            correct_train += (predicted == targets).sum().item()

        train_loss /= len(train_loader.dataset)
        train_accuracy = 100 * correct_train / len(train_loader.dataset)
        train_errors.append(train_loss)
        train_accuracies.append(train_accuracy)
        
        # Evaluation on test set
        model.eval()
        test_loss = 0.0
        correct_test = 0
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                batch_X = batch_X.to(device)
                batch_y = batch_y.to(device)
                outputs = model(batch_X)
                loss = loss_fn(outputs.squeeze(), batch_y)
                test_loss += loss.item() * batch_X.size(0)
                # Calculate accuracy
                predicted = torch.argmax(outputs.squeeze(), dim=1)
                targets = torch.argmax(batch_y, dim=1)
                correct_test += (predicted == targets).sum().item()

        test_loss /= len(test_loader.dataset)
        test_accuracy = 100 * correct_test / len(test_loader.dataset)
        test_errors.append(test_loss)
        test_accuracies.append(test_accuracy)

        tqdm_epoch.set_description(f"Epoch {epoch+1}/{epochs} \
            - Train loss: {train_loss:.4f}, Test loss: {test_loss:.4f}, \
            Train Acc: {train_accuracy:.2f}%, Test Acc: {test_accuracy:.2f}%")

    history = dict()
    history['train_loss'] = train_errors
    history['test_loss'] = test_errors
    history['train_acc'] = train_accuracies
    history['test_acc'] = test_accuracies
        
    return history

### Build AlexNet Model

This is the same structure as AlexNet with the input modified to use a 256 $\times$ 256 input and an output Softmax later with 10 elements for the 10 categories.

In [ ]:
class AlexNet(nn.Module):
    def __init__(self):
        super(AlexNet, self).__init__()
        self.conv1 = nn.Conv2d(3, 96, kernel_size=11, stride=4, padding=2)
        self.bn1 = nn.BatchNorm2d(96)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.conv2 = nn.Conv2d(96, 256, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm2d(256)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.conv3 = nn.Conv2d(256, 384, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(384)
        
        self.conv4 = nn.Conv2d(384, 384, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(384)
        
        self.conv5 = nn.Conv2d(384, 256, kernel_size=3, padding=1)
        self.bn5 = nn.BatchNorm2d(256)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.fc1 = nn.Linear(256 * 7 * 7, 4096)
        self.bn6 = nn.BatchNorm1d(4096)
        self.dropout1 = nn.Dropout(0.4)
        
        self.fc2 = nn.Linear(4096, 4096)
        self.bn7 = nn.BatchNorm1d(4096)
        self.dropout2 = nn.Dropout(0.4)
        
        self.fc3 = nn.Linear(4096, 1000)
        self.bn8 = nn.BatchNorm1d(1000)
        self.dropout3 = nn.Dropout(0.4)
        
        self.fc4 = nn.Linear(1000, 10)
        self.bn9 = nn.BatchNorm1d(10)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool1(x)
        
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool2(x)
        
        x = F.relu(self.bn3(self.conv3(x)))
        
        x = F.relu(self.bn4(self.conv4(x)))
        
        x = F.relu(self.bn5(self.conv5(x)))
        x = self.pool3(x)
        
        x = x.view(-1,  256 * 7 * 7)  # Flatten the tensor
        x = F.relu(self.bn6(self.fc1(x)))
        x = self.dropout1(x)
        
        x = F.relu(self.bn7(self.fc2(x)))
        x = self.dropout2(x)
        
        x = F.relu(self.bn8(self.fc3(x)))
        x = self.dropout3(x)
        
        x = F.softmax(self.bn9(self.fc4(x)), dim=1)
        
        return x

In [ ]:
alexnet = AlexNet().to(device)

In [ ]:
# Define the optimizer (Adam in this case)
optimizer = optim.Adam(alexnet.parameters(), lr=0.001)

# Define the loss function (categorical cross-entropy in this case)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
no_epochs = 5

In [ ]:
history_dict = dict()

### Train a model with Data Aumentation

In [ ]:
history = train_model(alexnet, train_loader, test_loader,
                      loss_fn, optimizer, no_epochs, device)

### Plot Validation Accuracy

In [ ]:
epochs = range(1, no_epochs + 1)

In [ ]:
fig, ax = plt.subplots(figsize=(7,5))

from cycler import cycler
monochrome = (cycler('color', ['k']) * cycler('linestyle', ['-', '--', ':', '-.']))

ax.set_prop_cycle(monochrome)
val_acc_values = history['test_acc']
ax.plot(epochs, val_acc_values, label = 'AlexNet Validation')
acc_values = history['train_acc']
ax.plot(epochs, acc_values, label = 'AlexNet Training')
    
ax.set_xlabel('Epochs')
ax.set_ylabel('Accuracy')
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
fig.savefig('TestAlexNet.png', dpi=300, bbox_inches='tight')